In [1]:
import numpy as np
from scipy.sparse import random, diags, csr_matrix
import Solve_Lanczos as lanczos

In [2]:
def generate_random_matrix(n, density=0.01):
    mat = random(n, n, density=density, format="csr", dtype=np.float64)
    row_sums = np.array(np.abs(mat).sum(axis=1)).flatten()
    diagonal_values = row_sums + 1.0
    diag_mat = diags(diagonal_values, 0)
    invertible_mat = mat + diag_mat
    return invertible_mat

In [3]:
N = 200
test = generate_random_matrix(N, density=0.04)
test = test.dot(test.T)

In [4]:
test_mat = test.toarray()
assert np.allclose(test_mat, test_mat.T, atol=1e-8), "Matrix is not symmetric"
assert np.all(np.linalg.eigvals(test_mat) > 0), "Matrix is not positive definite"

In [19]:
x0 = np.ones(N)
max_iter = 20
num_cores = 1
num_eigenvals = 2
find_max = True
result = lanczos.solve_lanczos(test.data, test.indices, test.indptr, x0, max_iter, num_eigenvals, find_max)

In [6]:
eigvals, eigvecs = np.linalg.eigh(test_mat)

In [20]:
for i in range(num_eigenvals):
    idx = i
    larger_smaller = "Smallest"
    if find_max:
        idx = -(idx+1)
        larger_smaller = "Largest"
    eigenval_error = abs(result[i][0] - eigvals[idx])
    eigenvec_error = np.linalg.norm(result[i][1] - eigvecs[:, idx])
    print(f"Result for {i + 1} {larger_smaller} Eigenpairs")
    print(f"Eigenvalue: Computed = {result[i][0]}, Actual = {eigvals[idx]}, Error = {eigenval_error}")
    print(f"Eigenvector Error = {eigenvec_error}")
    print(f"Eigenvector Error after sign correction: {np.linalg.norm(-result[i][1] - eigvecs[:, idx])}")

Result for 1 Largest Eigenpairs
Eigenvalue: Computed = 119.91707788710578, Actual = 119.91707788710575, Error = 2.842170943040401e-14
Eigenvector Error = 2.513894909278893e-08
Eigenvector Error after sign correction: 1.9999999999999996
Result for 2 Largest Eigenpairs
Eigenvalue: Computed = 102.6757842337143, Actual = 102.6758213895463, Error = 3.715583200403216e-05
Eigenvector Error = 0.0012536313213102697
Eigenvector Error after sign correction: 1.9999996071020891
